In [9]:
import pandas as pd

# === Étape 1 : ton ground truth depuis zeromev ===
ground_truth = sandwiches_zero_mev.copy()

print("=== Aperçu ground truth ===")
print(ground_truth.head())

# === Étape 2 : préparer detected_sandwiches ===
df_pred = detected_sandwiches.rename(columns={
    "block": "block_number",
    "front_idx": "front_tx_index",
    "victim_idx": "victim_tx_index",
    "back_idx": "back_tx_index",
    "front_value_in": "front_user_swap_volume",
    "victim_value_in": "victim_user_swap_volume",
    "back_value_in": "back_user_swap_volume"
})[[
    "block_number",
    "front_tx_index",
    "front_user_swap_volume",
    "victim_tx_index",
    "victim_user_swap_volume",
    "back_tx_index",
    "back_user_swap_volume"
]]

# Convertir les colonnes de volumes en float (elles étaient en object dans detected_sandwiches)
for col in ["front_user_swap_volume", "victim_user_swap_volume", "back_user_swap_volume"]:
    df_pred[col] = pd.to_numeric(df_pred[col], errors="coerce")

print("\n=== Aperçu detected harmonisé ===")
print(df_pred.head())

# === Étape 3 : Vérification des colonnes ===
print("\nColonnes ground_truth :", list(ground_truth.columns))
print("Colonnes df_pred      :", list(df_pred.columns))

missing_in_pred = set(ground_truth.columns) - set(df_pred.columns)
missing_in_gt = set(df_pred.columns) - set(ground_truth.columns)

if not missing_in_pred and not missing_in_gt:
    print("✅ Les colonnes correspondent parfaitement.")
else:
    print("⚠️ Différences détectées :")
    print(" - Dans ground_truth mais pas dans df_pred :", missing_in_pred)
    print(" - Dans df_pred mais pas dans ground_truth :", missing_in_gt)

# === Étape 4 : Comparaison (sur les indices uniquement) ===
keys = ["block_number", "front_tx_index", "victim_tx_index", "back_tx_index"]

set_gt = set(tuple(x) for x in ground_truth[keys].values)
set_pred = set(tuple(x) for x in df_pred[keys].values)

true_positives = set_gt & set_pred
false_positives = set_pred - set_gt
false_negatives = set_gt - set_pred

print("\n=== Résultats comparaison ===")
print(f"✅ True Positives : {len(true_positives)}")
print(f"❌ False Positives: {len(false_positives)}")
print(f"❌ False Negatives: {len(false_negatives)}")

# DataFrames pour inspection
df_tp = pd.DataFrame(list(true_positives), columns=keys)
df_fp = pd.DataFrame(list(false_positives), columns=keys)
df_fn = pd.DataFrame(list(false_negatives), columns=keys)

print("\nExemple de True Positives :")
print(df_tp.head())
print("\nExemple de False Positives :")
print(df_fp.head())
print("\nExemple de False Negatives :")
print(df_fn.head())

# === Étape 5 : Metrics ===
precision = len(true_positives) / (len(true_positives) + len(false_positives)) if (len(true_positives) + len(false_positives)) > 0 else 0
recall = len(true_positives) / (len(true_positives) + len(false_negatives)) if (len(true_positives) + len(false_negatives)) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("\n=== Metrics ===")
print(f"🎯 Précision : {precision:.4f}")
print(f"📈 Rappel    : {recall:.4f}")
print(f"📊 F1-score  : {f1:.4f}")


=== Aperçu ground truth ===
   block_number  front_tx_index  front_user_swap_volume  victim_tx_index  \
0      21948294               0                     NaN                4   
1      21948296              72                     NaN               73   
2      21948298               0                     0.0                1   
3      21948301               3                     NaN                4   
4      21948303               0                   374.6                1   

   victim_user_swap_volume  back_tx_index  back_user_swap_volume  
0                    88.24              7                 269.84  
1                    85.36             74                    NaN  
2                   622.70              2                   0.00  
3                   128.05              5                    NaN  
4                   135.16              4                 242.12  

=== Aperçu detected harmonisé ===
   block_number  front_tx_index  front_user_swap_volume  victim_tx_index  \
0 